# 01 — Quantum circuits, entanglement, and noise

**Phase 1 | Qiskit foundation**  
Implement small quantum circuits, compare exact state probabilities against sampled measurements, and quantify the effect of a controlled noise channel. The notebook uses only local simulators and seed **31**; it does not require an IBM Quantum account.

**Run:** select the `Python (QuantumOptLab)` kernel in VS Code, then **Run All**. Launch VS Code from the project root so saved reports use the correct paths.

### 1. Import the simulation tools

Load Qiskit, Aer, statevector utilities, numerical analysis, and plotting. Fix the shot budget and random seed so results can be reproduced.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import qiskit
import qiskit_aer
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

SEED = 31
SHOTS = 4096
simulator = AerSimulator()

figure_dir = Path("reports/figures")
experiment_dir = Path("reports/experiments")
figure_dir.mkdir(parents=True, exist_ok=True)
experiment_dir.mkdir(parents=True, exist_ok=True)

print(f"Qiskit {qiskit.__version__} | Aer {qiskit_aer.__version__} | seed={SEED}")


### 2. Verify single-qubit gates

The X gate maps $|0\rangle$ to $|1\rangle$. The H gate prepares equal measurement probabilities for $|0\rangle$ and $|1\rangle$. Check the exact statevector before measuring it in finite shots.

In [ ]:
x_circuit = QuantumCircuit(1)
x_circuit.x(0)
x_prob = Statevector.from_instruction(x_circuit).probabilities()
assert np.allclose(x_prob, [0.0, 1.0])

h_circuit = QuantumCircuit(1)
h_circuit.h(0)
h_prob = Statevector.from_instruction(h_circuit).probabilities()
assert np.allclose(h_prob, [0.5, 0.5])

h_measured = h_circuit.copy()
h_measured.measure_all()
h_counts = simulator.run(h_measured, shots=SHOTS, seed_simulator=SEED).result().get_counts()
p1 = h_counts.get("1", 0) / SHOTS
assert abs(p1 - 0.5) < 0.05

print("Hadamard circuit:\n", h_measured.draw(output="text"))
print("Exact probabilities:", h_prob)
print("Measured counts:", h_counts)
print(f"Measured P(1): {p1:.4f}")


### 3. Prepare and verify a Bell state

An H gate followed by CNOT produces $(|00\rangle+|11\rangle)/\sqrt{2}$. Check the exact amplitudes and ensure ideal measurements produce only `00` and `11`. Measurement counts are correlated, but the statevector establishes the prepared superposition.

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)

bell_state = Statevector.from_instruction(bell)
target_bell = np.array([1, 0, 0, 1], dtype=complex) / np.sqrt(2)
assert np.allclose(bell_state.data, target_bell)

bell_measured = bell.copy()
bell_measured.measure_all()
clean_counts = simulator.run(
    bell_measured, shots=SHOTS, seed_simulator=SEED
).result().get_counts()
assert set(clean_counts).issubset({"00", "11"})
assert 0.45 <= clean_counts.get("00", 0) / SHOTS <= 0.55

print(bell_measured.draw(output="text"))
print("Bell amplitudes:", bell_state.data)
print("Ideal measurement counts:", clean_counts)


### 4. Extend entanglement to three qubits

Prepare a GHZ state $(|000\rangle+|111\rangle)/\sqrt{2}$ using two CNOTs, then verify the two expected measurement strings.

In [ ]:
ghz = QuantumCircuit(3)
ghz.h(0)
ghz.cx(0, 1)
ghz.cx(1, 2)

ghz_prob = Statevector.from_instruction(ghz).probabilities_dict()
assert np.isclose(ghz_prob["000"], 0.5)
assert np.isclose(ghz_prob["111"], 0.5)
assert set(k for k, v in ghz_prob.items() if v > 1e-12) == {"000", "111"}

ghz_measured = ghz.copy()
ghz_measured.measure_all()
ghz_counts = simulator.run(
    ghz_measured, shots=SHOTS, seed_simulator=SEED
).result().get_counts()
assert set(ghz_counts).issubset({"000", "111"})
print(ghz_measured.draw(output="text"))
print("GHZ counts:", ghz_counts)
print("Note: Qiskit displays classical bits with the highest index on the left.")


### 5. Compare parameterized circuits with theory

For $R_y(\theta)|0\rangle$, the probability of measuring `1` is $\sin^2(\theta/2)$. Vary the angle and compare analytical and sampled probabilities. Parameterized gates will reappear when we implement QAOA.

In [ ]:
angles = np.linspace(0, np.pi, 9)
parameter_rows = []

for idx, theta in enumerate(angles):
    circuit = QuantumCircuit(1, 1)
    circuit.ry(float(theta), 0)
    circuit.measure(0, 0)
    measured = simulator.run(
        circuit, shots=SHOTS, seed_simulator=SEED + idx
    ).result().get_counts()
    observed_p1 = measured.get("1", 0) / SHOTS
    theoretical_p1 = np.sin(theta / 2) ** 2
    assert abs(observed_p1 - theoretical_p1) < 0.05
    parameter_rows.append({
        "theta_rad": float(theta),
        "theory_p1": float(theoretical_p1),
        "measured_p1": observed_p1,
    })

parameter_df = pd.DataFrame(parameter_rows)
display(parameter_df.round(4))


### 6. Add a controlled two-qubit noise channel

Attach a 12% depolarizing channel to the CNOT gate of the Bell circuit. Under ideal simulation, the measurement strings `01` and `10` should never appear; with this noise model, they can occur. This is a **synthetic noise model**, not a measured hardware calibration.

In [ ]:
noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(depolarizing_error(0.12, 2), ["cx"])
noisy_simulator = AerSimulator(noise_model=noise_model)

noisy_counts = noisy_simulator.run(
    bell_measured, shots=SHOTS, seed_simulator=SEED
).result().get_counts()

clean_parity_error = (
    clean_counts.get("01", 0) + clean_counts.get("10", 0)
) / SHOTS
noisy_parity_error = (
    noisy_counts.get("01", 0) + noisy_counts.get("10", 0)
) / SHOTS

assert clean_parity_error == 0.0
assert noisy_parity_error > 0.02

print("Ideal counts:", clean_counts)
print("Noisy counts:", noisy_counts)
print(f"Ideal parity-error fraction: {clean_parity_error:.4f}")
print(f"Noisy parity-error fraction: {noisy_parity_error:.4f}")


### 7. Run a reproducible noise sweep

Repeat the Bell experiment across several depolarizing strengths. Save a compact CSV and figure under `reports/`. The quantity plotted is the fraction of **odd-parity measurements** (`01` or `10`), which is a simple diagnostic rather than a complete state-fidelity metric.

In [ ]:
noise_strengths = [0.00, 0.02, 0.05, 0.10, 0.15, 0.20]
noise_rows = []

for strength in noise_strengths:
    if strength == 0:
        backend = simulator
    else:
        current_model = NoiseModel()
        current_model.add_all_qubit_quantum_error(
            depolarizing_error(strength, 2), ["cx"]
        )
        backend = AerSimulator(noise_model=current_model)

    counts = backend.run(
        bell_measured, shots=SHOTS, seed_simulator=SEED
    ).result().get_counts()
    error_count = counts.get("01", 0) + counts.get("10", 0)
    noise_rows.append({
        "depolarizing_strength": strength,
        "shots": SHOTS,
        "odd_parity_count": error_count,
        "odd_parity_fraction": error_count / SHOTS,
    })

noise_df = pd.DataFrame(noise_rows)
assert noise_df.iloc[0]["odd_parity_count"] == 0
assert noise_df.iloc[-1]["odd_parity_fraction"] > 0.05

csv_path = experiment_dir / "phase01_bell_noise_sweep_seed31.csv"
noise_df.to_csv(csv_path, index=False)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(noise_df["depolarizing_strength"], noise_df["odd_parity_fraction"], marker="o")
ax.set(xlabel="Two-qubit depolarizing strength", ylabel="Odd-parity fraction",
       title="Bell-state measurement errors under synthetic noise")
ax.grid(alpha=0.25)
fig.tight_layout()
figure_path = figure_dir / "phase01_bell_noise_sweep_seed31.png"
fig.savefig(figure_path, dpi=170, bbox_inches="tight")
plt.show()

print(f"Saved table: {csv_path}")
print(f"Saved figure: {figure_path}")
display(noise_df)


### 8. Verify the foundation checks

Confirm the exact-state, finite-shot, entanglement, parameterized-gate, and synthetic-noise checks completed without an assertion failure.

In [ ]:
checks = {
    "X gate": bool(np.allclose(x_prob, [0, 1])),
    "H gate": bool(np.allclose(h_prob, [0.5, 0.5])),
    "Bell amplitudes": bool(np.allclose(bell_state.data, target_bell)),
    "Bell ideal measurements": set(clean_counts).issubset({"00", "11"}),
    "GHZ ideal measurements": set(ghz_counts).issubset({"000", "111"}),
    "Ry sampled vs theory": bool((parameter_df["measured_p1"] - parameter_df["theory_p1"]).abs().max() < 0.05),
    "Noise diagnostic": noisy_parity_error > clean_parity_error,
    "Experiment table saved": csv_path.exists(),
    "Figure saved": figure_path.exists(),
}
assert all(checks.values()), checks
print(f"PASS: {sum(checks.values())}/{len(checks)} foundation checks")
